# On-policy distillation

**Notebook 04 · Let the student write, then learn from a frozen teacher**

An inspectable experiment with `Qwen/Qwen3-0.6B-Base` and `Qwen/Qwen3-32B`.
We start from the checkpoint saved in 03, compare two ways of continuing its
education, and measure all three resulting models on the same held-out questions.

> **Execution status:** Implemented, with local CPU checks recorded in
> [the validation record](assets/04_on_policy_distillation/VALIDATION.md).
> Real Qwen distillation and any resulting accuracy change remain unverified until
> this notebook is run in `train` mode on the Colab GPU. No performance curves are invented.

[1 · Goal](#goal) · [2 · Setup](#setup) · [3 · Data](#data) ·
[4 · Loss](#loss) · [5 · Starting student](#starting) · [6 · SFT control](#control) ·
[7 · Teacher and alignment](#teacher) · [8 · Distillation](#training) ·
[9 · Results](#results) · [10 · Save and recap](#save)

![The on-policy feedback loop](assets/04_on_policy_distillation/on_policy_workflow.png)

*The teacher scores the student's actual prefixes. It does not replace the response with its own answer.*

---
<a id="goal"></a>
## 1. Understand the experiment

Think of a student solving a math problem on paper. In 03, we supplied a worked
solution and taught the student to predict its words. Here, the student writes
its **own** solution. After each partial answer, the teacher tells us how likely
each possible next token should be. A token is a small piece of text, sometimes
a whole word and sometimes only part of one.

The teacher is **frozen**: its weights never change. Only the small student learns.
The teacher's probabilities are guidance about text, not a guarantee that the
arithmetic is correct. A teacher can still prefer a wrong continuation.

| Starting point | Additional practice | Purpose |
| :--- | :--- | :--- |
| **Starting SFT** | None | Measure the checkpoint we already have |
| **Continued SFT** | 1,024 new reference solutions | Check what more ordinary supervised practice achieves |
| **On-policy KD** | The same 1,024 questions, student-written responses, teacher distributions | Test learning on prefixes the student actually visits |

Both training arms start from **identical checkpoint bytes**, use the same question
order, and make 128 optimizer updates with eight examples per update. Their token
counts, compute cost, and learning signals differ. This compares two methods;
it does not isolate on-policy sampling from every other design choice.

![Three arms from the same checkpoint](assets/04_on_policy_distillation/controlled_comparison.png)

### How this relates to the paper

We implement the fully on-policy case of [GKD, Section 3.1](https://arxiv.org/html/2306.13649v3#S3.SS1):
sample from the current student, hold those token IDs fixed, and match the teacher's
next-token distributions on those prefixes. We use **forward KL** and teacher
temperature `0.1`, following its arithmetic experiment choices. Our Qwen models,
reference-solution warm start, prompt, data budget, and training scale are different.
This is an educational adaptation, not the paper's reported benchmark experiment.

**On-policy** describes where responses come from, not the direction of KL.
01 used reverse KL to inspect mechanics. 04 uses forward KL for this experiment.
No rewards, answer filtering, PPO, LoRA, or teacher-generated replacement solutions are used.

---
<a id="setup"></a>
## 2. Prepare the runtime and saved checkpoint

Start with `MODE = "inspect"` to check real data, token boundaries, and the math
without loading model weights. Choose `"train"` for the complete GPU experiment.

The unquantized 32B teacher needs an **A100 80GB** with most memory free. An L4
or A100 40GB is insufficient for this fixed recipe. The notebook checks memory
and disk before loading weights. It preserves Colab's installed PyTorch/CUDA.

In VS Code, select your Colab kernel and run from the top. The full project must
also exist on that remote server. A local notebook tab does not automatically
upload its `src/` helpers. Dependency changes require one kernel restart.
Restart the Python kernel before training to release models left over from 03;
this keeps the runtime's saved files while freeing GPU memory.

In [1]:
import importlib.metadata
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

# Inspection is safe on a CPU. Set "train" for your Colab A100 80GB experiment.
# The validator sets OPD_TRAIN_MODE=inspect in its separate local kernel.
MODE = os.environ.get("OPD_TRAIN_MODE", "inspect")
if MODE not in {"inspect", "train"}:
    raise ValueError("Choose MODE='inspect' or MODE='train'.")
IN_COLAB = bool(importlib.util.find_spec("google") and importlib.util.find_spec("google.colab"))
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/baseline.py").is_file()), None
)
if PROJECT_ROOT is None:
    if not IN_COLAB:
        raise RuntimeError("Open Jupyter inside the project checkout.")
    PROJECT_ROOT = Path("/content/on-policy-distillation")
    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "dev",
                "https://github.com/igoldin670/on-policy-distillation.git",
                str(PROJECT_ROOT),
            ],
            check=True,
        )
if not (PROJECT_ROOT / "src/opd.py").is_file():
    raise RuntimeError(
        "The remote checkout predates 04. Sync/upload the complete 04 project to this server, then restart the kernel."
    )

if IN_COLAB:
    # Use uv for dependencies. pip is used only to bootstrap uv if it is absent.
    # Never install a CPU torch wheel into the CUDA-enabled Colab environment.
    packages = [
        line.strip()
        for line in (PROJECT_ROOT / "requirements-generation.txt").read_text().splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]
    packages += [
        line.strip()
        for line in (PROJECT_ROOT / "requirements-notebook.txt").read_text().splitlines()
        if line.split("==")[0] in {"numpy", "pandas", "matplotlib"}
    ]
    packages += ["bottleneck==1.5.0", "ipywidgets==8.1.8", "PyYAML==6.0.3"]
    changed = []
    for requirement in packages:
        name, version = requirement.split("==")
        try:
            installed = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            installed = None
        module_name = {"PyYAML": "yaml"}.get(name, name)
        loaded = sys.modules.get(module_name)
        if loaded is not None and getattr(loaded, "__version__", installed) != installed:
            raise RuntimeError(f"Restart the kernel: {name} has mixed loaded/installed versions.")
        if installed != version:
            changed.append(requirement)
    if changed:
        if importlib.util.find_spec("uv") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run(
            [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable, *packages],
            check=True,
        )
        raise RuntimeError("Dependencies installed. Restart the kernel and run from the top.")

# Public pinned downloads do not need a Hugging Face secret or Google Drive.
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache/source-verification"))
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "opd-matplotlib"))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
runtime_project_label = PROJECT_ROOT if MODE == "train" else PROJECT_ROOT.name
print(f"Mode: {MODE} | Colab: {IN_COLAB} | Runtime project: {runtime_project_label}")

Mode: inspect | Colab: False | Runtime project: on-policy-distillation


In [2]:
import hashlib
import json
import math
import time
from dataclasses import asdict

import pandas as pd
import torch
import yaml
from datasets import disable_progress_bars, load_dataset
from IPython.display import display
from transformers import AutoConfig, AutoTokenizer, get_linear_schedule_with_warmup

from src.baseline import verify_recorded_run
from src.data import prepare_opd_splits
from src.generation import sample_trajectory
from src.logprobs import response_prediction_logits, selected_log_probs, verify_shared_tokenizer
from src.opd import (
    TEACHER_ID,
    TEACHER_REVISION,
    OPDConfig,
    append_trajectories,
    backward_opd_group,
    backward_sft_group,
    cuda_preflight,
    experiment_signature,
    export_report,
    fixed_trajectory_diagnostics,
    load_student,
    load_teacher,
    new_run_directory,
    optimizer_update,
    paired_accuracy_difference,
    read_resume,
    release_cuda_memory,
    resolve_cold_start,
    restore_training_state,
    save_resume,
    seed_everything,
    trajectory_tensors,
    truncate_uncommitted_trajectories,
    validate_trajectory_log,
)
from src.opd_loss import forward_kl_loss, forward_kl_per_token
from src.opd_plots import plot_run
from src.sft import (
    SFTConfig,
    collate_examples,
    encode_example,
    evaluation_summary,
    generate_evaluation,
    load_checkpoint_tokenizer,
    precision_context,
    prepare_splits,
    save_checkpoint,
    validation_loss,
    verify_checkpoint,
)
from src.utils import sha256_file, verify_inventory, write_json

# None means: find the newest complete Notebook 03 run on this runtime.
# If the old runtime is gone, upload both ZIPs, then set their SERVER-SIDE paths.
# Example: Path('/content/report.zip') and Path('/content/checkpoint.zip').
SFT_REPORT_PATH = None
SFT_CHECKPOINT_PATH = None

# To resume after interruption, set this to the existing 04 run directory.
# Matching optimizer-boundary snapshots are discovered for each arm separately.
RESUME_RUN_DIRECTORY = None

config = OPDConfig()
seed_everything(config.seed)
# Fixed seeds make random selections/draws repeatable on the same setup.
# Different GPU/software versions can still produce numerical differences.
disable_progress_bars()
baseline_evidence = verify_recorded_run(
    PROJECT_ROOT / "notebooks/assets/02_baseline_evaluation/recorded_run"
)
protocol = baseline_evidence["manifest"]["protocol"]
model_choices = yaml.safe_load((PROJECT_ROOT / "configs/small_experiment.yaml").read_text())
assert model_choices["student_model_id"] == protocol["student_model_id"]
assert model_choices["teacher_model_id"] == TEACHER_ID
lineage = json.loads(
    (
        PROJECT_ROOT / "notebooks/assets/04_on_policy_distillation/cold_start_lineage.json"
    ).read_text()
)
if MODE == "train":
    # A verified report is not a substitute for the actual trained student weights.
    sft_report_directory, starting_checkpoint_directory, sft_evidence = resolve_cold_start(
        PROJECT_ROOT, report_path=SFT_REPORT_PATH, checkpoint_path=SFT_CHECKPOINT_PATH
    )
    assert sft_evidence["protocol"] == protocol, "Keep 02/03's exact source and prompt protocol."
    runtime = cuda_preflight(PROJECT_ROOT)
    tokenizer = load_checkpoint_tokenizer(starting_checkpoint_directory)
    cold_splits = sft_evidence["splits"]
    checkpoint_hashes = sft_evidence["checkpoint_hashes"]
    print("Verified starting checkpoint:", starting_checkpoint_directory)
    print("GPU:", runtime["gpu"], "| memory GiB:", round(runtime["gpu_total_gib"], 1))
    print("BF16 supported:", runtime["bf16_supported"])
else:
    # Reconstruct the saved 03 split with real pinned data; no Qwen weights load.
    tokenizer = AutoTokenizer.from_pretrained(
        protocol["student_model_id"], revision=protocol["student_revision"], token=False
    )
    checkpoint_hashes = lineage["checkpoint_hashes"]
    print("INSPECT: tokenizer/data/math only. No teacher or student weights loaded.")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Fixed recipe:", asdict(config))
print("Optimizer updates per arm:", config.optimizer_steps)
print("Teacher revision:", TEACHER_REVISION)

INSPECT: tokenizer/data/math only. No teacher or student weights loaded.
Fixed recipe: {'seed': 42, 'train_examples': 1024, 'test_examples': 256, 'accumulation_steps': 8, 'learning_rate': 5e-06, 'weight_decay': 0.01, 'max_grad_norm': 1.0, 'warmup_fraction': 0.1, 'validation_interval': 32, 'checkpoint_interval': 32, 'max_prompt_length': 512, 'max_response_length': 256, 'max_sequence_length': 1024, 'teacher_temperature': 0.1, 'kl_chunk_size': 16, 'diagnostic_examples': 8}
Optimizer updates per arm: 128
Teacher revision: 9216db5781bf21249d130ec9da846c4624c16137


---
<a id="data"></a>
## 3. Keep practice, progress checks, and the exam separate

We use 1,024 **new** GSM8K training questions. They exclude every 03 practice and
validation question, and normalized question-text overlap with the official test
split. Continued SFT sees their cleaned reference solutions. On-policy KD receives
only their question prompts when it generates responses.

The same 64 validation questions from 03 are progress checks. The primary exam
contains 256 official test questions that were **not** in the 32-question subset
already inspected in 02/03. That earlier subset is reported separately as historical
context. We evaluate fixed final checkpoints, with no checkpoint selection or
hyperparameter tuning against either test set.

The candidate order and every exclusion are recorded. Long reference solutions
are skipped before training, never cut off mid-answer. Test selection uses the
question and prompt length, not how easy the answer looks.

In [3]:
# The TRAIN and TEST source revisions are pinned. Downloading data is not training.
source_train = load_dataset(
    protocol["dataset_id"],
    protocol["dataset_config"],
    split="train",
    revision=protocol["dataset_revision"],
    token=False,
)
source_test = load_dataset(
    protocol["dataset_id"],
    protocol["dataset_config"],
    split="test",
    revision=protocol["dataset_revision"],
    token=False,
)
if MODE == "inspect":
    cold_splits = prepare_splits(
        source_train, tokenizer, protocol, SFTConfig(**lineage["configuration"])
    )
    assert [r["source_index"] for r in cold_splits["train"]] == lineage["train_source_indices"]
    assert [r["source_index"] for r in cold_splits["validation"]] == lineage[
        "validation_source_indices"
    ]

# Check the saved text-to-token boundaries again with the tokenizer used NOW.
# This also detects tokenizer regex changes that vocabulary-size checks miss.
for record in cold_splits["train"] + cold_splits["validation"]:
    source_record = source_train[record["source_index"]]
    assert source_record["question"] == record["question"]
    encoded = encode_example(tokenizer, record["question"], source_record["answer"], protocol, 1024)
    assert encoded["input_ids"] == record["input_ids"]
    assert encoded["labels"] == record["labels"]
splits = prepare_opd_splits(
    source_train,
    source_test,
    tokenizer,
    protocol,
    cold_splits,
    train_examples=config.train_examples,
    test_examples=config.test_examples,
    seed=config.seed,
    max_prompt_length=config.max_prompt_length,
    max_sequence_length=config.max_sequence_length,
)
train_examples = splits["train"]
validation_examples = splits["validation"]
display(
    pd.DataFrame(
        [
            {"split": name, "questions": len(splits[name]), "role": role}
            for name, role in [
                ("train", "optimizer practice"),
                ("validation", "progress only"),
                ("primary_test", "final paired exam"),
                ("historical_test", "previously inspected context"),
            ]
        ]
    )
)
print("Selection exclusions recorded:", len(splits["exclusions"]))
print("One practice question:\n", train_examples[0]["question"])
print(
    "\nIts cleaned reference is used ONLY by the SFT control:\n", train_examples[0]["target_text"]
)

,split,questions,role
0,train,1024,optimizer practice
1,validation,64,progress only
2,primary_test,256,final paired exam
3,historical_test,32,previously inspected context


Selection exclusions recorded: 326
One practice question:
 Martin eats 1/2 cup of berries every day.  The grocery store is selling a package of berries (1 cup per pack) for $2.00.  How much will he spend on berries in a 30 day period?

Its cleaned reference is used ONLY by the SFT control:
 Martin eats 1/2 cup berries every day so over 30 days that's .5*30 = 15 cups of berries
The grocery store sells 1 cup packages of berries for $2.00 a pack.
Martin needs 15 cups so that's 2*15 = $30.00.
#### 30


### 3.1 What exactly receives a loss?

`input_ids` contains prompt tokens followed by response tokens. For SFT, the
response comes from the reference solution. For on-policy KD, it comes from the
student. Prompt tokens provide context but receive no training loss.

We process one unpadded response at a time during distillation. This removes the
need to guess which repeated end-token IDs are padding. SFT still uses explicit
`-100` labels to ignore prompt/padding positions. A real end token remains a target.

![The one-token prediction relationship](assets/04_on_policy_distillation/token_alignment.png)

If the prompt has `P` tokens, the first response token is at complete-sequence
index `P`, but its prediction is at logits index `P - 1`. The last prompt token's
logits predict what comes next. We therefore keep `logits[:, P - 1:-1, :]`.
The final logits position predicts an unobserved token and is excluded.

In [4]:
# A real reference example makes the boundary visible before any model is loaded.
example = train_examples[0]
prompt_length = example["prompt_length"]
preview_batch = collate_examples(train_examples[:2], tokenizer.pad_token_id)
print(
    "SFT tensors [batch_size, padded_sequence_length]:",
    {key: tuple(value.shape) for key, value in preview_batch.items()},
)
boundary_positions = range(prompt_length - 2, prompt_length + 3)
display(
    pd.DataFrame(
        [
            {
                "input_position": position,
                "role": "prompt" if position < prompt_length else "reference response",
                "token_id": example["input_ids"][position],
                "decoded_token": repr(tokenizer.decode([example["input_ids"][position]])),
                "training_label": example["labels"][position],
                "predicting_logits_position": position - 1,
            }
            for position in boundary_positions
        ]
    )
)
for row_index, record in enumerate(train_examples[:2]):
    complete_length = len(record["input_ids"])
    assert (preview_batch["labels"][row_index, : record["prompt_length"]] == -100).all()
    assert preview_batch["labels"][row_index, complete_length - 1].item() == tokenizer.eos_token_id
    assert (preview_batch["labels"][row_index, complete_length:] == -100).all()
print("Prompt masks, response boundaries, real EOS targets, and padding checks passed.")

SFT tensors [batch_size, padded_sequence_length]: {'input_ids': (2, 169), 'attention_mask': (2, 169), 'labels': (2, 169)}


,input_position,role,token_id,decoded_token,training_label,predicting_logits_position
0,78,prompt,16141,'Answer',-100,77
1,79,prompt,510,':\n',-100,78
2,80,reference response,49119,'Martin',49119,79
3,81,reference response,49677,' eats',49677,80
4,82,reference response,220,' ',220,81


Prompt masks, response boundaries, real EOS targets, and padding checks passed.


---
<a id="loss"></a>
## 4. Understand the learning signal

### 4.1 One answer token versus all possible next choices

Suppose a prefix ends with `3 + 2 =`. In a **made-up three-token vocabulary**:

| Next token | Teacher's probability | Student's probability |
| :--- | ---: | ---: |
| `5` | 80% | 10% |
| `6` | 15% | 70% |
| `7` | 5% | 20% |

SFT would say, "the reference token is `5`; make that token less surprising."
Forward KL instead compares both complete lists. The teacher strongly favors
`5`, while the student favors `6`. Learning should move the student's list toward
the teacher's. This is what **comparing possible next choices** means.

The real models have 151,936 output columns, not three. We keep all columns.
The sampled token's log probability is useful for inspection, but is **not**
the full-vocabulary KL loss.

### 4.2 The equation and the Python operation

For one student-written prefix, let $q$ be the teacher's target distribution and
$p_\theta$ the student's distribution. The temperature $T$ changes how concentrated
the teacher target is:

$$
q(v)=\operatorname{softmax}(z_T/T)_v,\qquad
p_\theta(v)=\operatorname{softmax}(z_S)_v.
$$

$$
D_{\mathrm{KL}}(q\Vert p_\theta)
=\sum_{v\in\mathcal V} q(v)\big[\log q(v)-\log p_\theta(v)\big].
$$

`forward_kl_per_token` implements this expression directly using FP32
`log_softmax`, multiplication, and a vocabulary sum. A natural logarithm gives
units called **nats**. Zero means identical distributions. Small negative values
near zero can arise from floating-point rounding; larger negatives are errors.

For a batch of $B$ sampled responses, response $i$ has $R_i$ tokens:

$$
\mathcal L_{\mathrm{OPD}}(\theta)
=\frac{1}{B}\sum_{i=1}^{B}\frac{1}{R_i}
\sum_{t=1}^{R_i} D_{\mathrm{KL}}\!\left(
q(\cdot\mid x_i,y_{i,<t})\Vert p_\theta(\cdot\mid x_i,y_{i,<t})\right).
$$

First average across each response, then across responses, as in GKD's sequence
divergence. A long answer does not outweigh a short answer just by being longer.
Both 04 arms use this per-response weighting. 03 used a pooled token mean, which
is a different valid weighting and remains unchanged there.

The related soft-target cross-entropy is $H(q,p)=-\sum_v q(v)\log p(v)$.
Since $D_{\mathrm{KL}}(q\Vert p)=H(q,p)-H(q)$ and the teacher is frozen,
both have the same student gradient on fixed prefixes. We report **actual KL**,
including the entropy term, rather than calling cross-entropy KL.

### 4.3 Two temperatures, two separate jobs

- **Student sampling temperature = 1.0:** draw responses from its current distribution.
  `top_k=0` and `top_p=1.0` remove truncation filters.
- **Teacher target temperature = 0.1:** sharpen the teacher's choices for this
  arithmetic experiment. This does not control student sampling. The student
  loss distribution stays at temperature 1; no extra $T^2$ multiplier is used.

We do not differentiate through the sampled token IDs. We run a new differentiable
student forward pass on their fixed prefixes. This is the GKD update, not a
policy-gradient estimate through the sampling process.

In [5]:
# These three probabilities are an ILLUSTRATION, not measurements from Qwen.
# Use teacher temperature 1 here because the table already specifies q directly.
toy_teacher_probs = torch.tensor([[[0.80, 0.15, 0.05]]])  # [1, 1, 3]
toy_student_probs = torch.tensor([[[0.10, 0.70, 0.20]]])
toy_kl = forward_kl_per_token(toy_student_probs.log(), toy_teacher_probs.log(), 1.0)
manual_kl = (toy_teacher_probs * (toy_teacher_probs.log() - toy_student_probs.log())).sum(-1)
assert torch.allclose(toy_kl, manual_kl, atol=1e-6)
assert toy_kl.item() > 0
print(f"Illustration only: teacher-to-student forward KL = {toy_kl.item():.4f} nats.")
print("No model weights were loaded or updated by this example.")

Illustration only: teacher-to-student forward KL = 1.3632 nats.
No model weights were loaded or updated by this example.


---
<a id="starting"></a>
## 5. Freeze the protocol and inspect the starting student

The 03 run saved real A100 evidence: validation format compliance increased from
0/64 to 59/64, strict numeric matches from 0/64 to 35/64, and exploratory prose
matches from 31/64 to 35/64. Its validation token loss fell from 0.776 to 0.667.
Five answers hit the token limit, and a displayed correct answer became incorrect.
Those are **historical 03 results**, not evidence that 04 improves anything.

04 makes one explicit stopping-policy change. The student uses `151643`
(`endoftext`) as its default end token; the teacher uses `151645` (`im_end`).
Both IDs have shared meanings. We stop at **either** ID for all three 04 arms,
including the unchanged starting model. All paired evaluations are rerun under
this rule, so they do not rely on a historical baseline with different settings.

Evaluation remains greedy, with a 512-token budget. Training rollouts are sampled
with a 256-token budget. A token-limit stop is a truncated prefix, not a successful
end: keep it, flag it, and do not append a fake EOS or reject it for being wrong.

In [6]:
# Reading teacher metadata/tokenizer does not download the 32B weight files.
teacher_config = AutoConfig.from_pretrained(TEACHER_ID, revision=TEACHER_REVISION, token=False)
teacher_tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_ID, revision=TEACHER_REVISION, token=False
)
student_config = AutoConfig.from_pretrained(
    protocol["student_model_id"], revision=protocol["student_revision"], token=False
)
assert student_config.vocab_size == teacher_config.vocab_size == 151936
tokenizer_checks = verify_shared_tokenizer(
    tokenizer,
    teacher_tokenizer,
    [
        text
        for row in train_examples + validation_examples
        for text in (row["prompt"], row["target_text"])
    ]
    + ["  spaces\n\n", "3 + 2 = 5.\n#### 5", "£12.50 − 3", "Café 中文 🙂"],
    teacher_config.vocab_size,
)
stop_token_ids = sorted({student_config.eos_token_id, teacher_config.eos_token_id})
evaluation_policy = dict(protocol["generation_config"])
evaluation_policy["eos_token_id"] = stop_token_ids
code_hashes = {
    str(path.relative_to(PROJECT_ROOT)): sha256_file(path)
    for path in sorted((PROJECT_ROOT / "src").glob("*.py"))
}
notebook_source = json.loads(
    (PROJECT_ROOT / "notebooks/04_on_policy_distillation.ipynb").read_text()
)
# Hash the teaching/experiment code, excluding editable runtime path/mode setup.
# Output cells and Markdown edits do not change the resume identity.
core_code = [
    (cell["id"], cell["source"])
    for cell in notebook_source["cells"]
    if cell["cell_type"] == "code" and cell["id"] not in {"opd-setup-runtime", "opd-configuration"}
]
code_hashes["notebook04_core_code"] = hashlib.sha256(
    json.dumps(core_code, ensure_ascii=False).encode()
).hexdigest()
signature = experiment_signature(
    config, protocol, splits, checkpoint_hashes, evaluation_policy, code_hashes
)
print("Shared tokenizer verified:", tokenizer_checks)
print("04 stopping IDs:", stop_token_ids)
print("Evaluation: greedy | max new tokens:", evaluation_policy["max_new_tokens"])

if MODE == "train":
    device, compute_dtype = torch.device("cuda:0"), torch.bfloat16
    run_directory = (
        Path(RESUME_RUN_DIRECTORY) if RESUME_RUN_DIRECTORY else new_run_directory(PROJECT_ROOT)
    )
    specification_path = run_directory / "experiment.json"
    if specification_path.exists():
        assert json.loads(specification_path.read_text())["signature"] == signature, (
            "Resume protocol differs. Start a new run."
        )
    else:
        write_json(
            specification_path,
            {
                "signature": signature,
                "configuration": asdict(config),
                "protocol": protocol,
                "teacher_model_id": TEACHER_ID,
                "teacher_revision": TEACHER_REVISION,
                "evaluation_policy": evaluation_policy,
                "stop_token_ids": stop_token_ids,
                "code_hashes": code_hashes,
                "sampling_policy": {
                    "do_sample": True,
                    "temperature": 1.0,
                    "top_k": 0,
                    "top_p": 1.0,
                    "max_new_tokens": config.max_response_length,
                    "eos_token_id": stop_token_ids,
                    "use_model_defaults": False,
                },
                "evaluation_use_model_defaults": False,
                "starting_checkpoint_hashes": checkpoint_hashes,
                "tokenizer_checks": tokenizer_checks,
            },
        )
        write_json(run_directory / "data_splits.json", splits)
        write_json(
            run_directory / "cold_start_evidence.json",
            {
                "manifest": sft_evidence["manifest"],
                "checkpoint_hashes": checkpoint_hashes,
                "configuration": json.loads(
                    (sft_report_directory / "configuration.json").read_text()
                ),
                "original_sft_splits": cold_splits,
                "report_checksums": json.loads(
                    (sft_report_directory / "SHA256SUMS.json").read_text()
                ),
            },
        )
    write_json(run_directory / "manifest.json", {"status": "in_progress", "signature": signature})
    print("04 artifacts on the runtime:", run_directory)
else:
    print(
        "INSPECT: source identity, new split isolation, full token mapping, and encoding checks passed."
    )
    inspection_directory = PROJECT_ROOT / "results/notebook04_qa"
    inspection_directory.mkdir(parents=True, exist_ok=True)
    write_json(
        inspection_directory / "inspection_facts.json",
        {
            "mode": "inspect",
            "split_counts": {
                name: len(splits[name])
                for name in ("train", "validation", "primary_test", "historical_test")
            },
            "selection_exclusions": len(splits["exclusions"]),
            "tokenizer_checks": tokenizer_checks,
            "stop_token_ids": stop_token_ids,
            "teacher_revision": TEACHER_REVISION,
            "maximum_selected_reference_length": max(
                len(row["input_ids"]) for row in train_examples
            ),
            "maximum_selected_prompt_length": max(row["prompt_length"] for row in train_examples),
            "model_weights_loaded": False,
        },
    )

Shared tokenizer verified: {'identical_token_mapping': True, 'text_probes_checked': 2180, 'output_vocabulary_size': 151936, 'student_eos_token_id': 151643, 'teacher_eos_token_id': 151645}
04 stopping IDs: [151643, 151645]
Evaluation: greedy | max new tokens: 512
INSPECT: source identity, new split isolation, full token mapping, and encoding checks passed.


In [7]:
# These helpers cache completed evaluations only inside this fixed-signature run.
# They retain every response, token ID, stop flag, and frozen checker result.
def evaluate_and_save(model, examples, name):
    from src.evaluation import rescore_response_rows

    path = run_directory / f"{name}.json"
    if path.exists():
        evaluation = json.loads(path.read_text())
        assert [r["example_id"] for r in evaluation["rows"]] == [r["example_id"] for r in examples]
        assert rescore_response_rows(evaluation["rows"])[1] == evaluation["scores"]
        return evaluation
    evaluation = generate_evaluation(
        model,
        tokenizer,
        examples,
        evaluation_policy,
        device,
        compute_dtype,
        progress_interval=16,
        use_model_defaults=False,
    )
    write_json(path, evaluation)
    return evaluation


if MODE == "train":
    starting_student = load_student(starting_checkpoint_directory, device)
    starting_validation_loss = validation_loss(
        starting_student, validation_examples, tokenizer.pad_token_id, device, compute_dtype
    )
    starting_validation = evaluate_and_save(
        starting_student, validation_examples, "starting_sft_validation"
    )
    display(pd.DataFrame([evaluation_summary(starting_validation, "starting_sft", "validation")]))
    print("Starting validation reference-token NLL:", starting_validation_loss)
    # A separate seed and forked RNG make fixed diagnostic responses reproducible
    # without consuming the random stream later used for on-policy training.
    diagnostic_path = run_directory / "fixed_validation_trajectories.json"
    if diagnostic_path.exists():
        diagnostic_trajectories = json.loads(diagnostic_path.read_text())
    else:
        with torch.random.fork_rng(devices=[device.index]):
            torch.manual_seed(config.seed + 10000)
            diagnostic_trajectories = [
                sample_trajectory(
                    starting_student,
                    tokenizer,
                    row,
                    device=device,
                    dtype=compute_dtype,
                    max_new_tokens=128,
                    stop_token_ids=stop_token_ids,
                    policy_step=0,
                )
                for row in validation_examples[: config.diagnostic_examples]
            ]
        write_json(diagnostic_path, diagnostic_trajectories)
    # Fixed validation trajectories are diagnostics only. They never enter training.
    del starting_student
    release_cuda_memory()
else:
    print("INSPECT: no baseline generation, model loss, or GPU performance measured.")

INSPECT: no baseline generation, model loss, or GPU performance measured.


---
<a id="control"></a>
## 6. Run the continued-SFT control

We first train a fresh copy of the 03 checkpoint on the new worked solutions.
This gives us a useful question: would ordinary additional practice have helped
as much as teacher feedback?

The visible loop below serves both arms. An **optimizer update** is one change
to the weights after eight examples have contributed gradients. On-policy KD
samples all eight responses **before** changing the student. Each response belongs
to that exact policy version and is used for one update only.

Defaults: FP32 student weights, BF16 autocast, gradient checkpointing, AdamW at
`5e-6`, weight decay `0.01`, gradient clipping at `1.0`, and a linear schedule
with 10% warmup. The control and KD use identical update counts and learning-rate
schedules. They are not matched in wall time or number of response tokens.

Every 32 updates we check validation and save a restart snapshot. Resume is allowed
only at a completed optimizer boundary with a matching experiment signature.
Two recent snapshots per arm are retained; final checkpoints are separate.

In [8]:
def train_arm(arm, teacher=None):
    from transformers import GenerationConfig

    # Every arm begins from the same checkpoint, unless resuming its own saved
    # optimizer state. No trained control weights become the KD starting point.
    assert arm in {"sft_control", "on_policy"}
    if arm == "on_policy" and teacher is None:
        raise ValueError("On-policy KD needs the frozen teacher.")
    final_directory = run_directory / "checkpoints" / arm
    result_path = run_directory / f"{arm}_training.json"
    if final_directory.exists() and result_path.exists():
        verify_inventory(final_directory)
        completed = json.loads(result_path.read_text())
        assert completed["completed_steps"] == config.optimizer_steps
        print(f"{arm}: reusing its verified completed checkpoint.")
        return completed

    snapshots = sorted((run_directory / "resume" / arm).glob("step_*"))
    resume_path = snapshots[-1] if snapshots else None
    resume_metadata = (
        read_resume(resume_path, arm=arm, signature=signature) if resume_path else None
    )
    model_source = resume_path / "student" if resume_path else starting_checkpoint_directory
    student = load_student(model_source, device)
    # Save the same inference/stopping policy used for this experiment into
    # every new checkpoint. The original 03 checkpoint stays untouched.
    student.generation_config = GenerationConfig.from_dict(evaluation_policy)
    student.generation_config.use_cache = True
    seed_everything(config.seed)
    optimizer = torch.optim.AdamW(
        student.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=math.ceil(config.optimizer_steps * config.warmup_fraction),
        num_training_steps=config.optimizer_steps,
    )
    # BF16 does not need scaling. The disabled scaler keeps one explicit loop API.
    scaler = torch.amp.GradScaler("cuda", enabled=False)
    history = resume_metadata["history"] if resume_metadata else []
    completed_steps = resume_metadata["completed_steps"] if resume_metadata else 0
    if resume_path:
        restore_training_state(resume_path, optimizer, scheduler, scaler)
    if teacher is not None:
        optimizer_ids = {id(p) for group in optimizer.param_groups for p in group["params"]}
        assert not optimizer_ids.intersection(id(p) for p in teacher.parameters())
        assert all(not p.requires_grad and p.grad is None for p in teacher.parameters())

    trajectory_path = run_directory / "training_trajectories.jsonl"
    if arm == "on_policy":
        truncate_uncommitted_trajectories(trajectory_path, completed_steps)
        validate_trajectory_log(
            trajectory_path, train_examples, completed_steps, config.accumulation_steps
        )
    optimizer.zero_grad(set_to_none=True)
    if torch.device(device).type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    started = time.perf_counter()
    for update_index in range(completed_steps + 1, config.optimizer_steps + 1):
        group_start = (update_index - 1) * config.accumulation_steps
        examples = train_examples[group_start : group_start + config.accumulation_steps]
        step_started = time.perf_counter()
        student.train()
        if arm == "on_policy":
            # Sampling uses no_grad. All trajectories use the same unchanged
            # weights, marked by the number of already completed updates.
            trajectories = [
                sample_trajectory(
                    student,
                    tokenizer,
                    example,
                    device=device,
                    dtype=compute_dtype,
                    max_new_tokens=config.max_response_length,
                    stop_token_ids=stop_token_ids,
                    policy_step=update_index - 1,
                )
                for example in examples
            ]
            metrics = backward_opd_group(
                student,
                teacher,
                trajectories,
                device,
                compute_dtype,
                config,
                scaler,
                policy_step=update_index - 1,
            )
        else:
            # Cross-entropy targets are reference response tokens; prompt/padding
            # targets are -100. Average response means across this actual group.
            metrics = backward_sft_group(
                student, examples, tokenizer.pad_token_id, device, compute_dtype, scaler
            )
        update_metrics = optimizer_update(
            student, optimizer, scheduler, scaler, config.max_grad_norm
        )
        if torch.device(device).type == "cuda":
            torch.cuda.synchronize()
        record = {
            "step": update_index,
            **metrics,
            **update_metrics,
            "update_seconds": time.perf_counter() - step_started,
        }
        record["peak_allocated_gpu_gib_so_far"] = (
            torch.cuda.max_memory_allocated() / 2**30
            if torch.device(device).type == "cuda"
            else None
        )
        if arm == "on_policy":
            append_trajectories(trajectory_path, trajectories)
            del trajectories  # No stale responses are reused at the next update.
        if update_index % config.validation_interval == 0 or update_index == config.optimizer_steps:
            record["validation_reference_nll"] = validation_loss(
                student, validation_examples, tokenizer.pad_token_id, device, compute_dtype
            )
            if teacher is not None:
                # Fixed validation prefixes let us compare like with like. The
                # training KL uses changing prefixes and need not decrease steadily.
                record["fixed_validation_kl"] = fixed_trajectory_diagnostics(
                    student, teacher, diagnostic_trajectories, device, compute_dtype, config
                )["mean_forward_kl_nats"]
        history.append(record)
        write_json(run_directory / f"{arm}_history.json", history)
        print(
            f"{arm} | update {update_index:3d}/{config.optimizer_steps} | loss {metrics['loss']:.4f} | response tokens {metrics['response_tokens']}",
            flush=True,
        )
        if update_index % config.checkpoint_interval == 0 or update_index == config.optimizer_steps:
            save_resume(
                run_directory,
                student,
                tokenizer,
                optimizer,
                scheduler,
                scaler,
                arm=arm,
                step=update_index,
                signature=signature,
                history=history,
            )

    # Free optimizer state before saving/reloading a second model for verification.
    if arm == "on_policy":
        validate_trajectory_log(
            trajectory_path, train_examples, config.optimizer_steps, config.accumulation_steps
        )
    elapsed = time.perf_counter() - started
    peak_gib = (
        torch.cuda.max_memory_allocated() / 2**30 if torch.device(device).type == "cuda" else None
    )
    del optimizer, scheduler, scaler
    release_cuda_memory()
    final_directory.parent.mkdir(parents=True, exist_ok=True)
    # A final checkpoint may already exist if export/verification was interrupted
    # after training. Verify it again instead of overwriting or retraining it.
    if final_directory.exists():
        verify_inventory(final_directory)
    else:
        save_checkpoint(student, tokenizer, final_directory)
    checkpoint_verified = verify_checkpoint(
        student,
        tokenizer,
        final_directory,
        validation_examples[0]["input_ids"],
        device,
        compute_dtype,
        probe_texts=["  spaces\n", "3 + 2 = 5.\n#### 5", "Café 中文 🙂"],
    )
    final_validation = evaluate_and_save(student, validation_examples, f"{arm}_validation")
    result = {
        "arm": arm,
        "completed_steps": len(history),
        "history": history,
        "current_session_loop_seconds": elapsed,
        "peak_allocated_gpu_gib": peak_gib,
        "summed_recorded_update_seconds": sum(row["update_seconds"] for row in history),
        "maximum_recorded_peak_gpu_gib": max(
            (
                row["peak_allocated_gpu_gib_so_far"]
                for row in history
                if row["peak_allocated_gpu_gib_so_far"] is not None
            ),
            default=None,
        ),
        "checkpoint_reload_verified": checkpoint_verified,
        "final_validation": evaluation_summary(final_validation, arm, "validation"),
        "resumed_from_step": completed_steps,
    }
    write_json(result_path, result)
    del student
    release_cuda_memory()
    return result

In [9]:
# The teacher is not needed for this reference-solution control.
if MODE == "train":
    control_result = train_arm("sft_control")
    display(pd.DataFrame([control_result["final_validation"]]))
else:
    print("INSPECT: training loop is defined, but no model has been trained.")

INSPECT: training loop is defined, but no model has been trained.


---
<a id="teacher"></a>
## 7. Load the teacher and prove token alignment

The teacher receives the **exact same complete token sequence** as the student:
the question prompt plus the student-written response. We do not decode and
retokenize the response, add a separate teacher chat template, or feed it a gold
reference. Each model predicts the next token after the same prefix.

The teacher is instruction-tuned, while the student is a base model with our SFT
warm start. Using the shared plain prompt preserves direct prefix comparison;
it differs from the teacher's recommended chat interface. We do not assume this
is the strongest way to elicit the teacher's reasoning.

`eval()` selects inference behavior; it does not freeze weights by itself.
`requires_grad_(False)` freezes teacher parameters, and `torch.no_grad()` prevents
building its gradient graph. `use_cache=False` avoids an unnecessary KV cache
when scoring a whole known sequence.

In [10]:
if MODE == "train":
    print(
        "Loading the frozen 32B teacher, about 61 GiB of weights. The first download can take several minutes.",
        flush=True,
    )
    teacher = load_teacher(device)
    student = load_student(starting_checkpoint_directory, device)
    student.eval()
    trajectory = diagnostic_trajectories[0]
    complete_sequence_ids, attention_mask = trajectory_tensors(trajectory, device)
    prompt_length = trajectory["prompt_length"]
    response_token_ids = complete_sequence_ids[:, prompt_length:]
    response_length = response_token_ids.shape[1]
    # Both forward passes use identical input IDs and masks: [1, complete_length].
    with torch.no_grad(), precision_context(device, compute_dtype):
        student_logits = student(
            input_ids=complete_sequence_ids, attention_mask=attention_mask, use_cache=False
        ).logits
        teacher_logits = teacher(
            input_ids=complete_sequence_ids, attention_mask=attention_mask, use_cache=False
        ).logits
    assert student_logits.shape == teacher_logits.shape
    assert student_logits.shape[:2] == complete_sequence_ids.shape
    print(
        "Logits [batch_size, complete_sequence_length, vocabulary_size]:",
        tuple(student_logits.shape),
    )
    student_response_logits = response_prediction_logits(
        student_logits, complete_sequence_ids, prompt_length
    )
    teacher_response_logits = response_prediction_logits(
        teacher_logits, complete_sequence_ids, prompt_length
    )
    # These selected scores describe the tokens that appeared. The KL below
    # separately compares ALL vocabulary choices at each response position.
    response_student_log_probs = selected_log_probs(student_response_logits, response_token_ids)
    response_teacher_target_log_probs = selected_log_probs(
        teacher_response_logits.float() / config.teacher_temperature, response_token_ids
    )
    assert (
        response_student_log_probs.shape
        == response_teacher_target_log_probs.shape
        == (1, response_length)
    )
    assert torch.equal(complete_sequence_ids[:, 1:][:, prompt_length - 1 :], response_token_ids)
    assert (
        torch.isfinite(student_response_logits).all()
        and torch.isfinite(teacher_response_logits).all()
    )
    assert (
        torch.isfinite(response_student_log_probs).all() and (response_student_log_probs <= 0).all()
    )
    assert (
        torch.isfinite(response_teacher_target_log_probs).all()
        and (response_teacher_target_log_probs <= 0).all()
    )
    smoke_kl = forward_kl_loss(
        student_response_logits,
        teacher_response_logits,
        teacher_temperature=config.teacher_temperature,
        chunk_size=config.kl_chunk_size,
        recompute=False,
    )
    print("One unchanged diagnostic response: forward KL =", float(smoke_kl), "nats")
    display(
        pd.DataFrame(
            [
                {
                    "final_prompt_token": repr(
                        tokenizer.decode([complete_sequence_ids[0, prompt_length - 1].item()])
                    ),
                    "first_response_token": repr(
                        tokenizer.decode([response_token_ids[0, 0].item()])
                    ),
                    "first_response_input_position": prompt_length,
                    "predicting_logits_position": prompt_length - 1,
                    "student_log_probability": response_student_log_probs[0, 0].item(),
                    "sharpened_teacher_target_log_probability": response_teacher_target_log_probs[
                        0, 0
                    ].item(),
                }
            ]
        )
    )
    display(
        pd.DataFrame(
            [
                {
                    "response_position": position,
                    "token_id": token_id,
                    "decoded_token": repr(tokenizer.decode([token_id])),
                    "student_log_probability": response_student_log_probs[0, position].item(),
                    "sharpened_teacher_target_log_probability": response_teacher_target_log_probs[
                        0, position
                    ].item(),
                }
                for position, token_id in enumerate(trajectory["response_token_ids"][:12])
            ]
        )
    )
    assert all(not p.requires_grad and p.grad is None for p in teacher.parameters())
    print(
        "Frozen teacher, identical prefixes, one-token shift, response count, and finite-value checks passed."
    )
    # Free the inspection tensors before training; retaining a notebook variable
    # can otherwise keep its large vocabulary tensor on the GPU.
    del student, student_logits, teacher_logits, student_response_logits, teacher_response_logits
    del response_student_log_probs, response_teacher_target_log_probs, smoke_kl
    release_cuda_memory()
    diagnostic_student = load_student(starting_checkpoint_directory, device)
    starting_fixed_diagnostics = fixed_trajectory_diagnostics(
        diagnostic_student, teacher, diagnostic_trajectories, device, compute_dtype, config
    )
    write_json(run_directory / "starting_sft_fixed_diagnostics.json", starting_fixed_diagnostics)
    del diagnostic_student
    release_cuda_memory()
else:
    print("INSPECT: teacher weights and real alignment logits still require the Colab GPU run.")

INSPECT: teacher weights and real alignment logits still require the Colab GPU run.


---
<a id="training"></a>
## 8. Learn on fresh student responses

Now the same loop starts **another fresh copy of the saved 03 checkpoint**.
For each group of eight questions, it samples eight responses, scores their
prefixes with the frozen teacher, backpropagates the response-mean KL, and makes
one student update. The next group uses the new student. Incorrect and truncated
responses stay in the experiment; there is no hidden answer-based rejection.

The teacher probabilities are detached. We recompute a differentiable student
forward pass instead of trying to backpropagate through `generate()`.
FP32 `log_softmax` improves numerical stability. Chunking response positions and
recomputing their softmax operations during backward reduces working memory while
preserving every vocabulary entry and the exact loss.

**Two different checks:** training KL describes freshly changing student prefixes.
KL on our eight **fixed validation responses** compares the same prefixes before
and after learning. The latter is a diagnostic, not another training set.
Neither number alone proves better generated answers.

In [11]:
if MODE == "train":
    # train_arm reloads the original saved SFT checkpoint, never control_result's weights.
    on_policy_result = train_arm("on_policy", teacher=teacher)
    assert all(not p.requires_grad and p.grad is None for p in teacher.parameters())
    display(pd.DataFrame([on_policy_result["final_validation"]]))
else:
    print("INSPECT: on-policy training has not run. No accuracy improvement is claimed.")

INSPECT: on-policy training has not run. No accuracy improvement is claimed.


---
<a id="results"></a>
## 9. Evaluate fixed final checkpoints

Only now do we open the primary held-out exam. Each checkpoint receives the
same questions, prompt, greedy decoding, end-token rules, token budget, and frozen
answer checkers. References are used **after generation** to score the final number.

- **Strict match:** correct numeric answer on the required `#### <number>` final line.
- **Format compliance:** that final line can be parsed, whether its number is right or wrong.
- **Exploratory prose match:** the unchanged broader extractor finds a matching final number.
- **Token-limit stops:** generation exhausted its budget without a recognized end token.

We keep every question in the denominator. A higher format score alone is not
better arithmetic. Numeric agreement also does not establish valid reasoning.
Paired bootstrap intervals summarize uncertainty across these questions, conditional
on one training seed. They do not measure run-to-run training variability.

The previously inspected 32-question subset is **secondary historical context**.
Its freshly generated 04 scores use 04's stopping rule and should not be presented
as a controlled comparison against old 02/03 aggregates.

In [12]:
if MODE == "train":
    # Score unchanged validation prefixes for the control and final KD checkpoint
    # while the teacher is still available. These are not optimizer inputs.
    fixed_diagnostics = {"starting_sft": starting_fixed_diagnostics}
    for arm in ("sft_control", "on_policy"):
        evaluated_student = load_student(run_directory / "checkpoints" / arm, device)
        fixed_diagnostics[arm] = fixed_trajectory_diagnostics(
            evaluated_student, teacher, diagnostic_trajectories, device, compute_dtype, config
        )
        del evaluated_student
        release_cuda_memory()
    write_json(run_directory / "fixed_diagnostics.json", fixed_diagnostics)
    display(
        pd.DataFrame(
            [
                {"checkpoint": arm, "fixed_prefix_forward_kl_nats": result["mean_forward_kl_nats"]}
                for arm, result in fixed_diagnostics.items()
            ]
        )
    )
    del teacher
    release_cuda_memory()
    print("Teacher released before the final generated-answer evaluations.")
else:
    print("INSPECT: no real fixed-prefix model comparison available yet.")

INSPECT: no real fixed-prefix model comparison available yet.


In [13]:
# Every arm receives the exact same held-out questions and decoding rules.
# The teacher is already released; no reference answer is passed to generate().
if MODE == "train":
    primary_evaluations, historical_evaluations = {}, {}
    checkpoint_directories = {
        "starting_sft": starting_checkpoint_directory,
        "sft_control": run_directory / "checkpoints/sft_control",
        "on_policy": run_directory / "checkpoints/on_policy",
    }
    for arm, checkpoint_directory in checkpoint_directories.items():
        evaluated_student = load_student(checkpoint_directory, device)
        primary_evaluations[arm] = evaluate_and_save(
            evaluated_student, splits["primary_test"], f"{arm}_primary_test"
        )
        historical_evaluations[arm] = evaluate_and_save(
            evaluated_student, splits["historical_test"], f"{arm}_historical_test"
        )
        del evaluated_student
        release_cuda_memory()
    primary_summary = pd.DataFrame(
        [
            evaluation_summary(result, arm, "primary_test")
            for arm, result in primary_evaluations.items()
        ]
    )
    display(primary_summary)
    print("Previously inspected subset, shown separately:")
    display(
        pd.DataFrame(
            [
                evaluation_summary(result, arm, "historical_test")
                for arm, result in historical_evaluations.items()
            ]
        )
    )
    paired_comparisons = {
        f"{first}_to_{second}": paired_accuracy_difference(
            primary_evaluations[first], primary_evaluations[second]
        )
        for first, second in [
            ("starting_sft", "sft_control"),
            ("starting_sft", "on_policy"),
            ("sft_control", "on_policy"),
        ]
    }
    display(pd.DataFrame(paired_comparisons).T)
    write_json(run_directory / "paired_comparisons.json", paired_comparisons)
    plot_run(
        {"sft_control": control_result["history"], "on_policy": on_policy_result["history"]},
        primary_evaluations,
        run_directory,
    )
    from IPython.display import Image

    display(Image(filename=str(run_directory / "training_objectives.png")))
    display(Image(filename=str(run_directory / "primary_test_comparison.png")))
else:
    print("INSPECT: final exam generation and performance figures await train mode.")

INSPECT: final exam generation and performance figures await train mode.


### 9.1 Read answers that changed

Aggregate scores can hide regressions. The next table includes both
incorrect-to-correct and correct-to-incorrect changes. Read their actual text
before deciding whether the model learned arithmetic, formatting, or a different
way of ending its answer. An empty table can be a valid result.

In [14]:
if MODE == "train":
    before = primary_evaluations["starting_sft"]
    after = primary_evaluations["on_policy"]
    before_rows = {r["example_id"]: r for r in before["rows"]}
    after_rows = {r["example_id"]: r for r in after["rows"]}
    before_scores = {r["example_id"]: r for r in before["annotations"]}
    changed_answers = []
    for row in after["annotations"]:
        key = row["example_id"]
        if before_scores[key]["strict_correct"] != row["strict_correct"]:
            changed_answers.append(
                {
                    "example_id": key,
                    "change": "improved" if row["strict_correct"] else "regressed",
                    "question": before_rows[key]["question"],
                    "expected_answer": before_rows[key]["expected_answer"],
                    "starting_response": before_rows[key]["response_text"],
                    "on_policy_response": after_rows[key]["response_text"],
                }
            )
    # Include both directions in the visible preview instead of cherry-picking gains.
    preview = [
        r
        for direction in ("improved", "regressed")
        for r in [v for v in changed_answers if v["change"] == direction][:2]
    ]
    display(pd.DataFrame(preview))
    write_json(run_directory / "changed_answers.json", changed_answers)
else:
    print("INSPECT: no generated-answer changes to inspect.")

INSPECT: no generated-answer changes to inspect.


---
<a id="save"></a>
## 10. Save the evidence and explain the outcome

The small report includes exact split IDs, sampled trajectories, raw evaluation
responses, loss histories, source code, configuration, model revisions, resource
measurements, and checksums. It excludes full weights and optimizer snapshots.
The final Hugging Face checkpoints are saved in separate directories and verified
by reload, tokenizer round trips, and logits comparison.
Their saved generation defaults include both recognized end-token IDs and the
greedy evaluation policy, so exported weights carry their stopping settings too.

A `/content/...` path is on **Colab's server**. Creating a ZIP there does not mean
it has downloaded to your computer. In a VS Code Colab connection, use its file
explorer or download action to copy the printed report and checkpoint paths.
Save the executed notebook too. Colab storage can disappear when the session ends.

No Google Drive or automatic large browser download is used. Set
`EXPORT_CHECKPOINT_ZIPS = True` if you want separate weight ZIPs created on the
server, then download them using VS Code.

In [15]:
# Weight archives are optional because each full student checkpoint is large.
EXPORT_CHECKPOINT_ZIPS = False
if MODE == "train":
    assert (
        control_result["checkpoint_reload_verified"]
        and on_policy_result["checkpoint_reload_verified"]
    )
    manifest = {
        "status": "completed",
        "method": "on_policy_forward_kl_with_sft_control",
        "signature": signature,
        "training_seed": config.seed,
        "student_model_id": protocol["student_model_id"],
        "student_revision": protocol["student_revision"],
        "teacher_model_id": TEACHER_ID,
        "teacher_revision": TEACHER_REVISION,
        "teacher_frozen": True,
        "teacher_temperature": config.teacher_temperature,
        "starting_checkpoint_hashes": checkpoint_hashes,
        "optimizer_steps_per_arm": config.optimizer_steps,
        "runtime": {
            **runtime,
            "torch": torch.__version__,
            "cuda": torch.version.cuda,
            "python": sys.version,
            "dataset_fingerprints": {
                "train": source_train._fingerprint,
                "test": source_test._fingerprint,
            },
            "huggingface_packages": {
                name: importlib.metadata.version(name)
                for name in [
                    "transformers",
                    "datasets",
                    "accelerate",
                    "huggingface_hub",
                    "numpy",
                    "pandas",
                    "matplotlib",
                    "ipywidgets",
                    "PyYAML",
                ]
            },
        },
        "limitations": [
            "One training seed",
            "Custom subset, not an official leaderboard score",
            "Prose checker is exploratory",
            "Numeric agreement does not verify reasoning",
            "Different losses and compute budgets across SFT and KD",
            "Plain shared prompt differs from the teacher's chat interface",
            "03/02 historical metrics used different end-token rules",
        ],
    }
    write_json(run_directory / "manifest.json", manifest)
    report_archive = export_report(PROJECT_ROOT, run_directory, manifest)
    print("REPORT ZIP ON COLAB:", report_archive)
    for arm in ("sft_control", "on_policy"):
        checkpoint_directory = run_directory / "checkpoints" / arm
        print(f"{arm} CHECKPOINT FOLDER ON COLAB:", checkpoint_directory)
        if EXPORT_CHECKPOINT_ZIPS:
            archive = shutil.make_archive(
                str(run_directory / f"{arm}_checkpoint"), "zip", root_dir=checkpoint_directory
            )
            print(f"{arm} CHECKPOINT ZIP ON COLAB:", archive)
    print("Export checks passed. Copy these artifacts and the executed notebook to your computer.")
else:
    print(
        "Inspection complete. No Qwen training, teacher scoring, or performance claim was made in this run."
    )

Inspection complete. No Qwen training, teacher scoring, or performance claim was made in this run.


### What we achieved, in the simplest terms

**01:** We followed one answer through generation, teacher scoring, probability
comparison, gradients, and one update. This checked the machinery, not better answers.

**02:** We measured the original student. We learned that failing to use the
requested answer format is different from getting the number wrong.

**03:** We taught the student from worked solutions and saved it. The real A100
run learned the format strongly; arithmetic results were mixed, with visible regressions.

**04:** We built a fair starting point for comparing additional worked examples
with teacher guidance on student-written responses. We made token boundaries,
loss direction, fresh sampling, frozen teacher weights, and evaluation rules explicit.

- **If you ran inspection:** you checked real data isolation, tokenizer compatibility,
  response masks, and an illustrative KL calculation. You did not train Qwen.
- **If you completed training:** you generated fresh student trajectories, updated
  only the student, ran the SFT comparison, evaluated the same final exam, and saved
  the actual results. Read those numbers before claiming improvement.

The teacher says, "after what you have written, here are the next choices I prefer."
The student learns from that advice. The held-out answers tell us whether it helped.

### What a stronger follow-up would need

Repeat the fixed experiment with additional training seeds, examine failures, and
reserve new held-out questions before any tuning. An off-policy KD arm with the
same KL would help isolate the effect of student-generated prefixes from the
effect of switching from hard reference targets to teacher distributions.
Teacher task performance under our shared plain prompt is also unmeasured.

### References

- [GKD paper: method, Algorithm 1, and arithmetic experiments](https://arxiv.org/abs/2306.13649)
- [Qwen3-32B model card](https://huggingface.co/Qwen/Qwen3-32B)
- [Pinned GSM8K source](https://huggingface.co/datasets/openai/gsm8k/tree/740312add88f781978c0658806c59bc2815b9866)
- [Previous notebook: supervised cold start](03_sft_cold_start.ipynb)
- [Implementation and validation limits](assets/04_on_policy_distillation/VALIDATION.md)